In [ ]:
import os
import sys
from pathlib import Path
import sys, time
import subprocess
import time
import requests
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# Make aai/ importable from ipy_nb/
sys.path.insert(0, "..")
from aai.prompts import (
    FILE_SUMMARIZER_PROMPT,
    CONTEXT_MANAGER_PROMPT,
    ARCHITECT_PROMPT,
    CRITIC_PROMPT,
)
from aai.repo_reader import TEXT_SUFFIXES

# --- CONFIGURATION ---
REPO_PATH = "../code_base"
OUTPUT_DIR = Path("output_analysis")
MAX_CHUNK_CHARS = 12000 # Roughly 3k-4k tokens. Change this depending on the model you use.

# Initialize Directories
STAGES = ["01_scout", "02_aggregate", "03_draft", "04_critique", "05_refined", "06_visual"]
for stage in STAGES:
    (OUTPUT_DIR / stage).mkdir(parents=True, exist_ok=True)

# Load environment variables
load_dotenv()

True

# TODO
changes I want to make....

More refactoring... Use document chunking and crawling from langchain. Lower priority...
Move the modules to a better file structure/use Manikandan's code base. This one should have some discussion, but I kind of like the idea of using classes because it's more stateful imo.

In [14]:
# list ollama models
# !ollama list

In [5]:
# list cached models (on mac hardware)
!ls -F ~/.cache/huggingface/hub/

In [15]:
# For macs, mlx-lm is more optimized on the hardware
# Default running a 4 bit quantized version
def start_mlx_server(model_id="mlx-community/Qwen2.5-7B-Instruct-4bit", port=8080):
    # 1. Check if server is already running
    try:
        requests.get(f"http://localhost:{port}/v1/models", timeout=2)
        print(f"Server already running on port {port}.")
        return None
    except:
        print(f"Starting MLX server with {model_id}...")
        # 2. Start the process
        cmd = ["python", "-m", "mlx_lm.server", "--model", model_id, "--port", str(port)]
        process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 3. Wait for readiness
        for _ in range(30):
            try:
                requests.get(f"http://localhost:{port}/v1/models", timeout=1)
                print("Server is ready!")
                return process
            except:
                time.sleep(2)
        print("Failed to start server.")
        return None


def get_model():
    """Returns a LangChain chat model based on LLM_PROVIDER env var.
    
    Supported providers (set LLM_PROVIDER in .env):
      - 'local'        : MLX local server via ChatOpenAI-compatible wrapper
      - 'local-ollama' : Ollama via ChatOllama
      - 'claude'       : Anthropic Claude Sonnet via ChatAnthropic
      - 'openai'       : OpenAI GPT-4o via ChatOpenAI (default)
    """
    provider = os.getenv("LLM_PROVIDER", "local").lower()

    if provider == "local":
        # Ensure MLX server is running: python -m mlx_lm.server --model <model_id>
        start_mlx_server()
        model_id = requests.get("http://localhost:8080/v1/models", timeout=5).json()["data"][0]["id"]
        return ChatOpenAI(
            model=model_id,
            base_url="http://localhost:8080/v1",
            api_key="local",
            temperature=0,
        )

    elif provider == "local-ollama":
        # Ensure you have run: ollama pull <MODEL_NAME>
        model_name = os.getenv("MODEL_NAME", "qwen3-vl:8b")
        return ChatOllama(
            model=model_name,
            base_url="http://localhost:11434",
            temperature=0,
        )

    elif provider == "claude":
        # Requires ANTHROPIC_API_KEY in .env
        # Optionally set ANTHROPIC_MODEL (default: claude-sonnet-4-5)
        model_name = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")
        return ChatAnthropic(
            model=model_name,
            api_key=os.getenv("ANTHROPIC_API_KEY"),
            temperature=0,
        )

    else:  # 'openai' or any other value falls back to OpenAI
        model_name = os.getenv("OPENAI_MODEL", "gpt-4o")
        return ChatOpenAI(
            model=model_name,
            api_key=os.getenv("OPENAI_API_KEY"),
            temperature=0,
        )


In [7]:
'''
# If you want to test your server/API
with client.chat.completions.create(
    model=model_id,  # Ensure you have run 'ollama pull llama3'
    messages=[{"role": "user", "content": "Explain why the sky is blue."}],
    stream=True,
) as response:
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)'''

In [16]:
class Agent:
    """Base class for all pipeline agents.
    
    Provides shared functionality:
      - save_md: write markdown output to the stage's output directory
      - collect_files: walk a directory and read all files into dicts
      - calculate_total_size: sum character counts across a list of strings
      - invoke_llm: send a system + human message pair to an LLM
    """

    def __init__(self, stage, debug=False):
        self.stage = stage
        self.debug = debug
        self.system_prompt = ""
        self.tolken_use = []

    def save_md(self, filename, data):
        """Write data to OUTPUT_DIR / self.stage / filename."""
        try:
            print(f"saving to {OUTPUT_DIR / self.stage / filename}")
            with open(OUTPUT_DIR / self.stage / f"{filename}.md", 'w') as f:
                f.write(data)
        except Exception as e:
            print(f"{e}: data")

    def collect_files(self, source_path, filter_fxn):
        """Walk source_path and return all readable files as a list of dicts.
        
        Each dict has keys 'path' (relative to source_path) and 'content'.
        Hidden directories (starting with '.') are skipped.
        """
        all_files = []
        for root, _, files in os.walk(source_path):
            if '/.' in root:
                continue
            for file in files:
                full_path = os.path.join(root, file)
                if filter_fxn(file):
                    try:
                        with open(full_path, 'r', encoding='utf-8') as f:
                            content = f.read()
                            all_files.append({
                                "path": os.path.relpath(full_path, source_path),
                                "content": content
                            })
                    except Exception as e:
                        print(f"Could not read {full_path}: {e}")
        return all_files
    
    def calculate_total_size(self, items):
        """Sum the character lengths of a list of strings."""
        return sum(len(s) for s in items)

    def invoke_llm(self, llm, human_content):
        """Send self.system_prompt + human_content to the LLM and return the response text."""
        from langchain_core.messages import SystemMessage, HumanMessage
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=human_content),
        ]
        response = llm.invoke(messages)
        # Track token usage
        self.tolken_use.append(response.usage_metadata)
        return response.content

In [17]:
# NOTE max_chars_per_chunk is very different depending on the setup...
# ~50k for a 4-bit quantized 8B model on an M4, ~500k for gpt-4o.
class FileSummarizer(Agent):
    """Stage 1 – walks a source repo, chunks files, and summarizes each chunk.
    
    File filtering uses TEXT_SUFFIXES from aai/repo_reader.py by default.
    Pass a custom list to `extensions` to override.
    """

    def __init__(self, repo_path, extensions=None, max_chars_per_chunk=50000, debug=False, stage=STAGES[0]):
        super().__init__(stage=stage, debug=debug)
        self.repo_path = repo_path
        self.extensions = extensions or list(TEXT_SUFFIXES)  # from aai/repo_reader.py
        self.max_chars_per_chunk = max_chars_per_chunk
        self.system_prompt = FILE_SUMMARIZER_PROMPT

    def _should_include(self, filename):
        """Return True if the file's extension is in self.extensions."""
        return any(filename.endswith(ext) for ext in self.extensions)

    def collect_files(self):
        """Walk self.repo_path and return only files matching self.extensions."""
        return super().collect_files(self.repo_path, self._should_include)

    def create_chunks(self, all_files):
        """Group files into chunks that fit within the LLM character limit."""
        chunks = []
        current_chunk = []
        current_size = 0

        for file_data in all_files:
            file_size = len(file_data['content'])
            if current_size + file_size > self.max_chars_per_chunk and current_chunk:
                chunks.append(current_chunk)
                current_chunk = []
                current_size = 0
            current_chunk.append(file_data)
            current_size += file_size

        if current_chunk:
            chunks.append(current_chunk)
        return chunks

    # NOTE make async def if calling asynchronously
    def summarize_chunk(self, chunk, llm):
        """Send a chunk of files to the LLM and return the structured summary."""
        formatted_content = "\n\n".join([
            f"--- FILE: {f['path']} ---\n{f['content']}"
            for f in chunk
        ])
        return self.invoke_llm(llm, formatted_content)

    def process_all(self, chunks, llm):
        """Summarize every chunk and save each result to the stage output dir."""
        start = time.time()
        processed_sum = []
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i}, with {len(chunk)} files")
            summary = self.summarize_chunk(chunk, llm)
            processed_sum.append(summary)
            print(f"saving summary{i} to {self.stage}")
            self.save_md(f"summary{i}", summary)
            print(f"took {time.time() - start:.1f} seconds")
            start = time.time()
        return processed_sum


In [10]:
# Map files into chunks that can be summarized
llm = get_model()
scout = FileSummarizer("../code_base", debug=True)

In [11]:
files = scout.collect_files()
chunks = scout.create_chunks(files)

In [ ]:
scout.process_all(chunks, llm)

In [21]:
system_prompt = """
        You are a system archetect. Give a summary of the given code files.
        Output a Markdown response with a YAML frontmatter header.
        Identify:
        1. Module Name (based on folder or primary file)
        2. Key Dependencies (imports/includes)
        3. Primary Responsibilities
        4. Public APIs or Interfaces used
        """

formatted_content = "\n\n".join([
    f"--- FILE: {f['path']} ---\n{f['content']}" 
    for f in chunks[0]
])

# NOTE this timed out after 30m with ollama on a 16g M4 and a 8b qwen model after trying on zookeeper chunk 0
from langchain_core.messages import SystemMessage, HumanMessage
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=formatted_content),
])

In [25]:
response

In [28]:
response.usage_metadata

In [13]:
task = scout.summarize_chunk(chunk=chunks[0], llm=llm)

In [22]:
'''# Recommended way to run multiple chunks concurrently
tasks = [FileSummarizer.summarize_chunk(c) for c in chunks]
results = await asyncio.gather(*tasks)'''

In [18]:
class ContextManager(Agent):
    """Stage 2 – reads scout summaries and recursively reduces them until they
    fit within the architect's context window.
    """

    def __init__(self, architect_threshold=20000, debug=False, stage=STAGES[1]):
        super().__init__(stage=stage, debug=debug)
        self.architect_threshold = architect_threshold
        self.scout_path = OUTPUT_DIR / STAGES[0]
        self.system_prompt = CONTEXT_MANAGER_PROMPT

    def collect_files(self):
        """Read all files from the scout output directory."""
        return super().collect_files(self.scout_path, lambda f: f.endswith(".md"))

    def summarize_summaries(self, summary_batch, llm):
        """Collapse a group of summaries into a single 'Module Overview'."""
        combined_text = "\n\n".join(summary_batch)
        return self.invoke_llm(llm, combined_text)

    def reduce(self, summaries, llm):
        """Recursively shrink summaries until the total size fits the threshold."""
        current_summaries = summaries

        while self.calculate_total_size(current_summaries) > self.architect_threshold:
            print(f"Current size {len(str(current_summaries))} exceeds threshold. Reducing...")

            new_summaries = []
            chunk_size = 5
            for i in range(0, len(current_summaries), chunk_size):
                batch = current_summaries[i : i + chunk_size]
                reduced_summary = self.summarize_summaries(batch, llm)
                new_summaries.append(reduced_summary)
                self.save_md(f"reduced_sum{i}", reduced_summary)

            current_summaries = new_summaries

        if self.calculate_total_size(current_summaries) < self.architect_threshold:
            print(f"Current size {len(str(current_summaries))} does not need reduction")
            self.save_md("reduced_sum", str(current_summaries))
        return current_summaries


In [34]:
# Map files into chunks that can be summarized
aggregator = ContextManager(debug=True)
files = aggregator.collect_files()
reduced_files = aggregator.reduce(files, llm)

In [22]:
class ArchitectAgent(Agent):
    """Stage 3/5 – produces and refines a Mermaid architecture diagram.

    draft(llm):
      - Reads aggregated summaries from 02_aggregate/
      - Optionally incorporates an external reference .md file
      - Produces an initial Mermaid diagram
      - Saves output to 03_draft/mermaid.md

    revise(llm, arch_md_path=None):
      - Reads the draft diagram from 03_draft/
      - Reads the critique from 04_critique/
      - Reads original aggregated context from 02_aggregate/ (source of truth)
      - Optionally incorporates an external reference .md file
      - Uses the same ARCHITECT_PROMPT (which includes the Iterative Refinement section)
      - Saves the refined output to 05_refined/mermaid.md
    """

    def __init__(self, arch_md_path=None, debug=False, stage=STAGES[2]):
        super().__init__(stage=stage, debug=debug)
        self.agg_path      = OUTPUT_DIR / STAGES[1]  # 02_aggregate/
        self.draft_path    = OUTPUT_DIR / STAGES[2]  # 03_draft/
        self.critique_path = OUTPUT_DIR / STAGES[3]  # 04_critique/
        self.arch_md_path  = arch_md_path            # optional external reference .md
        self.system_prompt = ARCHITECT_PROMPT

    def collect_files(self):
        """Read all .md files from the aggregator output directory."""
        return super().collect_files(self.agg_path, lambda f: f.endswith(".md"))

    def load_arch_md(self):
        """Optionally load an external architecture .md file as reference context.

        Returns the file content as a string, or None if no path was provided.
        """
        if self.arch_md_path:
            try:
                with open(self.arch_md_path, 'r', encoding='utf-8') as f:
                    return f.read()
            except Exception as e:
                print(f"Could not read arch_md_path {self.arch_md_path}: {e}")
        return None

    def draft(self, llm):
        """Generate the initial Mermaid architecture diagram from aggregated summaries.

        Reads all .md files from 02_aggregate/, combines them, and invokes the LLM.
        Saves the result to 03_draft/mermaid.md.
        """
        self.stage = STAGES[2]  # 03_draft
        summaries = self.collect_files()
        arch_md   = self.load_arch_md()

        if not summaries:
            print("No aggregated summaries found in 02_aggregate/. Run ContextManager first.")
            return None

        parts = []
        agg_text = "\n\n".join(
            f"--- FILE: {f['path']} ---\n{f['content']}" for f in summaries
        )
        parts.append(f"## Aggregated Summaries\n{agg_text}")

        if arch_md:
            parts.append(f"## Reference Architecture (.md file)\n{arch_md}")

        human_content = "\n\n---\n\n".join(parts)

        print(f"Running ArchitectAgent draft on {len(summaries)} summary file(s)...")
        diagram = self.invoke_llm(llm, human_content)
        self.save_md("mermaid", diagram)
        return diagram

    def revise(self, llm, arch_md_path=None):
        """Refine the Mermaid diagram using critic feedback.

        Builds a human message containing:
          - Original aggregated summaries (02_aggregate/) as source of truth
          - The current draft Mermaid diagram (03_draft/)
          - The critic's structured feedback (04_critique/)
          - An optional external reference .md file

        Saves the refined diagram to 05_refined/mermaid.md.
        """
        self.stage = STAGES[4]  # 05_refined
        if arch_md_path:
            self.arch_md_path = arch_md_path

        summaries = self.collect_files()  # 02_aggregate/
        drafts    = super().collect_files(self.draft_path,    lambda f: f.endswith(".md"))
        critiques = super().collect_files(self.critique_path, lambda f: f.endswith(".md"))
        arch_md   = self.load_arch_md()

        if not drafts:
            print("No Mermaid diagram found in 03_draft/. Run draft() first.")
            return None
        if not critiques:
            print("No critique found in 04_critique/. Run CritiqueAgent first.")
            return None

        parts = []

        if summaries:
            agg_text = "\n\n".join(
                f"--- FILE: {f['path']} ---\n{f['content']}" for f in summaries
            )
            parts.append(f"## Original Aggregated Summaries (Source of Truth)\n{agg_text}")

        draft_text = "\n\n".join(
            f"--- FILE: {f['path']} ---\n{f['content']}" for f in drafts
        )
        parts.append(f"## Current Mermaid Diagram (Draft)\n{draft_text}")

        critique_text = "\n\n".join(
            f"--- FILE: {f['path']} ---\n{f['content']}" for f in critiques
        )
        parts.append(f"## Critic Feedback\n{critique_text}")

        if arch_md:
            parts.append(f"## Reference Architecture (.md file)\n{arch_md}")

        human_content = "\n\n---\n\n".join(parts)

        print(f"Running ArchitectAgent revision with {len(critiques)} critique(s) "
              f"and {len(summaries)} summary file(s)...")
        refined = self.invoke_llm(llm, human_content)
        self.save_md("mermaid", refined)
        return refined


In [23]:
# Run the ArchitectAgent — initial draft
# arch_md_path is optional — pass a reference .md file for additional context
# e.g. arch_md_path="../zookeeper_architecture.md"
architect = ArchitectAgent(arch_md_path="../zookeeper_architecture.md", debug=True)

In [ ]:
diagram = architect.draft(llm)
print(diagram)

In [20]:
class CritiqueAgent(Agent):
    """Stage 4 – reviews the Mermaid diagram from the ArchitectAgent and produces
    structured critique feedback to be used as input for the next revision cycle.

    Inputs (read automatically):
      - 03_draft/*.md  : The Mermaid diagram(s) produced by ArchitectAgent
      - 02_aggregate/*.md : Subsystem summaries for cross-referencing claims
      - arch_md_path (optional): An external reference architecture .md file
        (e.g. zookeeper_architecture.md) for the critic to investigate

    Output:
      - 04_critique/critique.md : Structured critique following the four-section
        format defined in prompts/critic-agent-v2.md:
          1. Identified Architectural Issues
          2. Edge & Relationship Actions
          3. Missing or Hidden Components
          4. Critic's Summary
    """

    def __init__(self, arch_md_path=None, debug=False, stage=STAGES[3]):
        super().__init__(stage=stage, debug=debug)
        self.draft_path = OUTPUT_DIR / STAGES[2]   # 03_draft/
        self.agg_path   = OUTPUT_DIR / STAGES[1]   # 02_aggregate/
        self.arch_md_path = arch_md_path           # optional external .md reference
        self.system_prompt = CRITIC_PROMPT

    def collect_files(self):
        """Read the Mermaid diagram(s) from the draft stage output directory."""
        return super().collect_files(self.draft_path, lambda f: f.endswith(".md"))

    def collect_context(self):
        """Read aggregated subsystem summaries for cross-referencing architectural claims."""
        return super().collect_files(self.agg_path, lambda f: f.endswith(".md"))

    def load_arch_md(self):
        """Optionally load an external architecture .md file for the critic to investigate.
        
        Returns the file content as a string, or None if no path was provided.
        """
        if self.arch_md_path:
            try:
                with open(self.arch_md_path, 'r', encoding='utf-8') as f:
                    return f.read()
            except Exception as e:
                print(f"Could not read arch_md_path {self.arch_md_path}: {e}")
        return None

    def critique(self, llm):
        """Run the Critic LLM on the Mermaid diagram + context and save the critique.
        
        Builds a structured human message containing:
          - Subsystem summaries (from 02_aggregate/) as repository context
          - The candidate Mermaid diagram (from 03_draft/)
          - An optional reference architecture .md file

        Returns the critique text and saves it to 04_critique/critique.md.
        """
        diagrams = self.collect_files()
        context  = self.collect_context()
        arch_md  = self.load_arch_md()

        if not diagrams:
            print("No Mermaid diagram found in 03_draft/. Run ArchitectAgent first.")
            return None

        # Build the human message with clearly labelled sections
        parts = []
        if context:
            subsystem_text = "\n\n".join(
                f"--- FILE: {f['path']} ---\n{f['content']}" for f in context
            )
            parts.append(f"## Subsystem Summaries\n{subsystem_text}")

        diagram_text = "\n\n".join(
            f"--- FILE: {f['path']} ---\n{f['content']}" for f in diagrams
        )
        parts.append(f"## Candidate Architecture (Mermaid Diagram)\n{diagram_text}")

        if arch_md:
            parts.append(f"## Reference Architecture (.md file)\n{arch_md}")

        human_content = "\n\n---\n\n".join(parts)

        print(f"Running CritiqueAgent on {len(diagrams)} diagram(s) "
              f"with {len(context)} context file(s)...")
        critique_text = self.invoke_llm(llm, human_content)
        self.save_md("critique", critique_text)
        return critique_text


In [ ]:
# Run the CritiqueAgent
# arch_md_path is optional — pass a reference .md file for the critic to investigate
# e.g. arch_md_path="../zookeeper_architecture.md" to cross-reference against the diagram
critic = CritiqueAgent(arch_md_path="../zookeeper_architecture.md", debug=True)
critique = critic.critique(llm)
print(critique)

In [24]:
# Run the ArchitectAgent — revision pass
# Reads: 02_aggregate/ (source of truth), 03_draft/ (current diagram), 04_critique/ (feedback)
# arch_md_path is optional — pass a reference .md file for additional context
# Saves refined diagram to 05_refined/mermaid.md
refined = architect.revise(llm)
print(refined)

Running ArchitectAgent revision with 1 critique(s) and 1 summary file(s)...
saving to output_analysis/05_refined/mermaid
Based on the provided summaries and the reference architecture, I will create a more accurate and evidence-based Mermaid diagram. The diagram will include the key components and their interactions as described in the summaries and the reference architecture.

```mermaid
graph TD
    %% ── Project Metadata ──────────────────────────────────────────────
    subgraph META["Project Infrastructure (confidence: 0.85)"]
        ASF[".asf.yaml\nProject Metadata"]
        README["README / README_packaging\nDocs & Build Instructions"]
        TYPOS[".typos.toml\nTypo Config"]
        MERGE["zk-merge-pr.py\nPR Automation (GitHub/JIRA)"]
        VERSION["Info.java / VersionInfoMain\nVersion Info"]
        COMPAT["check_compatibility.py\nJava API Compat Check"]
    end

    %% ── ZooKeeper Server Core (inferred from test targets) ────────────
    subgraph SERVER["ZooKeeper Server

In [ ]:
# --- Stage 6: Render Mermaid diagrams to PNG ---
# Requires: pip install mmdc

import re
from mmdc import MermaidConverter

VISUAL_DIR = OUTPUT_DIR / STAGES[5]  # 06_visual/

def extract_mermaid(md_content):
    """Strip markdown fences and return raw Mermaid syntax."""
    match = re.search(r"```mermaid\s*(.*?)```", md_content, re.DOTALL)
    return match.group(1).strip() if match else md_content.strip()

def render_to_png(md_path, out_png):
    """Read a Mermaid .md file and save it as a PNG using mermaid-py."""
    content = Path(md_path).read_text(encoding="utf-8")
    mermaid_src = extract_mermaid(content)
    
    converter = MermaidConverter()
    converter.to_png(mermaid_src, str(out_png))
    print(f"✅ Saved: {out_png}")

# Render draft and refined diagrams
for stage_idx, label in [(2, "draft"), (4, "refined")]:  # 03_draft, 05_refined
    src = OUTPUT_DIR / STAGES[stage_idx] / "mermaid.md"
    if src.exists():
        render_to_png(src, VISUAL_DIR / f"mermaid_{label}.png")
    else:
        print(f"⚠️  Not found: {src}")

✅ Saved: output_analysis/06_visual/mermaid_draft.png
✅ Saved: output_analysis/06_visual/mermaid_refined.png
